# Case Study - X-ray Photoelectron Spectroscopy (XPS)

<div class="alert alert-block alert-info">
<strong>Questions:</strong>

<ul>
    <li>How can I use Python to fit analyze XPS data?</li>
</ul>

<strong>Objectives:</strong>

<ul>
    <li>Use pandas to read XPS data.</li>
    <li>Use SciPy to find peaks.</li>
    <li>Create interactive plots data.</li>
    <li>Perform a background subtraction.</li>
    <li>Find and  fit peaks with Gaussian functions.</li>
</ul>
</div>


XPS is a surface-sensitive quantitative technique that measures the elemental composition, empirical formula, chemical state, and electronic state of elements present in a material. 
It works on the principle of photoemission, where an X-ray beam ejects core-level electrons from the surface of the sample, and the kinetic energy of these electrons is measured.

In this notebook, we will process [XPS spectra of silicon](https://pubs.aip.org/avs/sss/article-abstract/20/1/36/366432/Silicon-100-SiO2-by-XPS?redirectedFrom=fulltext).
We will use many of the skills learned in previous notebooks including processing data using pandas and visualization using plotly.
This notebook introduces a new library - SciPy. 
SciPy is an advanced Python library tailored for scientific computing and analysis.
We will use SciPy for peak fitting and integration.

Note that this notebook is **not** a best practices for XPS analysis - rather it shows you Python strategies that can be used for data processing and analysis.

For this notebook, we will be using `pandas`, `plotly`, `scipy`, and `numpy`.

We will follow these steps:

1. Analyze wide scan spectrum - We will read in the wide scan and use SciPy `find_peaks` to find peaks.
2. Analyze narrow spectra - For the oxygen and silicon peaks, we will do a more in-depth analysis.

For the narrow spectra, we will:
1. Load in the data.
2. Perform a background correction using a polynomial fit to the background.
3. Fit peaks in the spectrum using a Gaussian function.
4. Integrate each peak.

**Instructor edition.** Completed examples and worked solutions. Run all cells from top to bottom using the project Python environment. Data are read from the parent repository.

In [1]:
from pathlib import Path
import sys

# Support kernels started in the repository root or in instructor-notebooks.
ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "data").is_dir():
    raise FileNotFoundError("Open this notebook from the repository or instructor-notebooks folder.")
sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

from scipy.signal import savgol_filter, find_peaks
from scipy.optimize import curve_fit
from scipy.integrate import simpson, quad

##  Reading in Data

In this notebook, we will again use `pandas` to read in our data.
For past notebooks, we have used the `read_csv` function. 
However, the data set we will use for this notebook is in a tab delimited file.

We could use `read_csv` with this data, and specify that the delimiter is a tab character.
However the [`read_table`](https://pandas.pydata.org/docs/reference/api/pandas.read_table.html) function is more general for this purpose.

When you examine the data for this notebook by viewing one of the files, you will see something like the following:

```
1.3500000e+003	6.5014286e+003
1.3495000e+003	6.4672997e+003
1.3490000e+003	6.4709445e+003
1.3485000e+003	6.4449666e+003
```

There are no headers for this data, so we will tell `pandas`that by putting in the argument `header=None`.

In [3]:
wide_scan = pd.read_table(ROOT / "si_data/N0125701.asc", header=None)
wide_scan.head()

,0,1
0,1350.0,6501.4286
1,1349.5,6467.2997
2,1349.0,6470.9445
3,1348.5,6444.9666
4,1348.0,6430.1221


Because there is no header, our columns do not yet have names.
We will rename the columns to `energy` and `intensity`.

In [4]:
wide_scan.columns = ["energy", "intensity"]
wide_scan.head()

,energy,intensity
0,1350.0,6501.4286
1,1349.5,6467.2997
2,1349.0,6470.9445
3,1348.5,6444.9666
4,1348.0,6430.1221


## Visualizing the Data

Upon loading the data initially, you would probably wish to examine it.
We will plot our data using plotly express and `px.line`.

If you look at the order of the energy data in the `wide_scan` dataframe above, you will see that the energy
is listed from highest to lowest. Typically XPS spectra are shown with the highest enregy on the left side of the axis.
In order to have our graph appear this way, we can use `fig.update_xaxes(autorange="reversed")`


<div class="alert alert-block alert-info">
<strong>Why do we reverse the x axis?</strong>

<p>
We reverse the x axis on XPS spectra so that the order of the peaks represent how tightly bound electrons are.
When the axis is reversed, more tightly bound electrons are on the left, and more loosely bound electrons are on the right.
</p>
</div>

In [5]:
# Create an interactive plot of the wide scan
fig = px.line(wide_scan, x="energy", y="intensity") # line plot of data
fig.update_xaxes(autorange="reversed")  # reverse x axis
fig.show()

## Data Smoothing

Sometimes data may be noisy and require smoothing. 
The data set we are working with does not require smoothing, but an example is shown below in case you need to smooth data!

In [6]:
smoothed = wide_scan.copy()
# Smooth only intensity, preserving measured energies.
smoothed["intensity"] = savgol_filter(wide_scan["intensity"], window_length=11, polyorder=3)

In [7]:
# Create an interactive plot of the wide scan
fig = px.line(smoothed, x="energy", y="intensity")
fig.update_xaxes(autorange="reversed")  
fig.show()

## Peak Finding

On our XPS spectrum, the position of peaks tells us what elements are present in the sample. 
In order to analyze our spectrum, we would want to find the peaks, then compare our peaks to peaks of known elements.

SciPy provides a [function called `find_peaks`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html) that we will use to automatically pick peaks from our spectrum. 
The documentation linked above describes the available arguments and includes examples.

We will use the `prominence` keyword to define how much a peak must stand out from its neighbors. This will prevent us from picking peaks based on noise.

In [8]:
# Fit the peaks of the wide scan
peak_index, properties = find_peaks(x=wide_scan["intensity"], prominence=1000)

In [9]:
print(peak_index)

[ 247  666  699  740 1326 1589 1633 1898 2128 2294 2329 2362 2390 2397
 2428 2465 2492 2500 2648 2683]


SciPy returns the *positions* of detected peaks within the input sequence, rather than their intensities or energies.
For example, consider these five intensity values:

| Position | Intensity |
| --- | --- |
| 0 | 1 |
| 1 | 3 |
| 2 | 7 |
| 3 | 3 |
| 4 | 1 |

For this example, `find_peaks` returns position `2`, where the intensity is `7`.
In our spectrum, we use `.iloc` with `peak_index` to select the rows containing the energy and intensity of each detected peak.


In [10]:
peaks = wide_scan.iloc[peak_index].copy()
peaks.reset_index(inplace=True, drop=True)
peaks.head()

,energy,intensity
0,1226.5,10368.952
1,1017.0,19281.154
2,1000.5,24225.567
3,980.0,35057.086
4,687.0,18176.167


We can now visualize our peak locations using plotly.

In [11]:
fig = px.line(wide_scan, x="energy", y="intensity")
fig.add_scatter(x=peaks["energy"], y=peaks["intensity"], 
                mode="markers", 
                name="peaks")
fig.update_xaxes(autorange="reversed")   
fig.show()

<div class="alert alert-block alert-warning">
<strong>Exercise</strong>

<p>
As you can see, just using the <code>find_peaks</code> function returns many more peaks to us than we would choose to label. 
</p>

<p>
<a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html">Read the documentation for the <code>find_peaks</code> function.</a>
</p>

<p>
Try different arguments in order to find appropriate peaks in the spectrum.
</p>

<p>
<strong>To continue with the tutorial, change your <code>find_peaks</code> function above to</strong>:
</p>

<pre><code>peak_index, properties = find_peaks(x=wide_scan[&quot;intensity&quot;], prominence=1000)</code></pre>
</div>

**Worked solution:** Compare prominence thresholds on the same measured spectrum. Larger thresholds retain peaks that stand out more strongly; distance is measured in samples. These are exploratory settings. Preserve the original `peak_index` and `peaks` for the labeling exercise.

In [12]:
for prominence in [1000, 3000, 10000]:
    trial_index, _ = find_peaks(wide_scan["intensity"], prominence=prominence, distance=10)
    print("Prominence:", prominence, "Peak count:", len(trial_index))
    display(wide_scan.iloc[trial_index])

Prominence: 1000 Peak count: 18


,energy,intensity
247,1226.5,10368.9520
666,1017.0,19281.1540
699,1000.5,24225.5670
740,980.0,35057.0860
1326,687.0,18176.1670
1589,555.5,29428.6980
1633,533.5,157753.3300
1898,401.0,20934.1300
2128,286.0,37744.8000
2294,203.0,27499.9730


Prominence: 3000 Peak count: 12


,energy,intensity
247,1226.5,10368.9520
699,1000.5,24225.5670
740,980.0,35057.0860
1589,555.5,29428.6980
1633,533.5,157753.3300
2128,286.0,37744.8000
2329,185.5,29156.9240
2362,169.0,32894.4250
2397,151.5,60643.5750
2465,117.5,20698.2750


Prominence: 10000 Peak count: 7


,energy,intensity
740,980.0,35057.086
1589,555.5,29428.698
1633,533.5,157753.330
2128,286.0,37744.800
2362,169.0,32894.425
2397,151.5,60643.575
2500,100.0,66897.220


## Peak Labeling

We might wish to add another column to our table that gives the peak identities.
We can first create an empty column in our data frame for peak identities.
We will start by creating a blank column

In [13]:
# Create a blanke column called "labels" with nothing in it.
peaks["labels"] = None

# Preview the first 20 rows of the dataframe.
peaks.head(20)

,energy,intensity,labels
0,1226.5,10368.9520,None
1,1017.0,19281.1540,None
2,1000.5,24225.5670,None
3,980.0,35057.0860,None
4,687.0,18176.1670,None
5,555.5,29428.6980,None
6,533.5,157753.3300,None
7,401.0,20934.1300,None
8,286.0,37744.8000,None
9,203.0,27499.9730,None


Next, we will fill in identities for some of the peaks. 
For example, from looking up a [reference table](https://www.thermofisher.com/us/en/home/materials-science/learning-center/periodic-table.html), we would know that the peak at 100 is $Si$, and the peak at 104 is $Si$ in $SiO_2$.

In [14]:
peaks.loc[17, "labels"] = "Si"
peaks.loc[16, "labels"] = "Si (SiO2)"

In [15]:
peaks.head(20)

,energy,intensity,labels
0,1226.5,10368.9520,None
1,1017.0,19281.1540,None
2,1000.5,24225.5670,None
3,980.0,35057.0860,None
4,687.0,18176.1670,None
5,555.5,29428.6980,None
6,533.5,157753.3300,None
7,401.0,20934.1300,None
8,286.0,37744.8000,None
9,203.0,27499.9730,None


<div class="alert alert-block alert-warning">
<strong>Exercise</strong>

<p>
Use the following <a href="https://www.thermofisher.com/us/en/home/materials-science/learning-center/periodic-table.html">reference table</a> to identify the oxygen peak.
</p>

<p>
After you have identified it, add a label to the appropriate row.
</p>
</div>

In [16]:
oxygen_row = (peaks["energy"] - 533).abs().idxmin()
peaks.loc[oxygen_row, "labels"] = "O 1s"
peaks.loc[[oxygen_row]]

,energy,intensity,labels
6,533.5,157753.33,O 1s


We can add labels to our graphs by adding a `text` and `textposition` argument to our plot.

Use the cell below to visualize your plot with labels.

You may need to change the range of the y-axis once you have added the oxygen plot. 
You can do so with the command

```python
fig.update_yaxes(range=[minimum_y_value, maximum_y_value])
```

In [17]:
fig = px.line(wide_scan, x="energy", y="intensity")
fig.add_scatter(x=peaks["energy"], y=peaks["intensity"], 
                mode="markers+text", 
                name="peaks",
                text=peaks["labels"],  # this was added  to add labels
                textposition="top center",  # this was added to position the labels.
               )
fig.update_xaxes(autorange="reversed")   
fig.show()

## Peak Analysis

A common practice in XPS analysis is to analyze particular peaks in order to determine the chemical states, concentrations, and elemental composition of a sample.
For the next section of this notebook, we will consider a close-up view of the Si peaks. 
We will fit and subtract the background signal, then we will fit Gaussian functions to the peaks.

In [18]:
# data loading
si_peaks = pd.read_table(ROOT / "si_data/N0125702.asc", header=None)

# add column names
si_peaks.columns = ["energy", "intensity"]

# visualize
si_fig = px.line(si_peaks, x="energy", y="intensity")
si_fig.update_xaxes(autorange="reversed")
si_fig.show()

### Background Subtraction

Upon viewing the figure above, we can see the presence of some background signal, particularly on the left side of the graph.
Using your mouse hovering over the graph to examine, you might decide that for the peak on the left the background corresponds to
energies less than 98.5 eV or greater than 106.5 eV.

In the cell below, we select parts of our signal that fit that criteria using

```
(98.5 > si_peaks["energy"]) | (si_peaks["energy"] > 106.5)
```

in this syntax, `|` represents "or". The phrase above says that the energy should be less than 98.5 or greater than 106.5. 
Similar to peak finding, this returns the index for where the statement is true.
We get the actual values by using the index with the peaks.

In [19]:
# define what we want to fit for background subtraction
background_index = (98.5 > si_peaks["energy"]) | (si_peaks["energy"] > 106.5)
background = si_peaks[background_index]
background.head()

,energy,intensity
0,112.000,3262.6982
1,111.875,3229.2760
2,111.750,3138.2208
3,111.625,3037.6669
4,111.500,3043.9405


We can visualize what we've selected for the background by adding more data to the figure we created previously.

In [20]:
si_fig.add_scatter(x=background["energy"], y=background["intensity"], mode="markers", name="background")
si_fig.show()

In order to subtract out this background, we will fit a polynomial function to our selected data then subtract it from the signal.
This will adjust our baseline to 0.

To do this in Python, we will use a [function from numpy called `polyfit`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html). `polyfit` is specifically for fitting polynomials of any order.
To use polyfit, you pass in your x values, your y values, and your desired polynomial degree.
For this particular case, it looks as though a linear fit (polynomial degree=1) will be sufficient. 
However, you could increase this or change it if you needed a different polynomial degree.

In [21]:
# Fit a polynomial of degree 1
fit = np.polyfit(background["energy"], background["intensity"], 1)

print(fit)

[   137.77256437 -12432.6279266 ]


What is returned to us in fit is the parameters for the fit. 
In this case, the first value is our slope and the second is the intercept.
We can pass this into another function `np.polyval` to evaluate the value of the polynomial at given energies.
We will simply evaluate the fit over the whole energy range for these peaks.

In [22]:
# Get fit values
vals = np.polyval(fit, si_peaks["energy"])

We can add this to our visualization to evaluate our fit.

In [23]:
si_fig.add_scatter(x=si_peaks["energy"], y=vals, name="background fit")
si_fig.show()

Now that we have determined the equation for our background, we can subtract it from our intensity to get our corrected peaks.

In [24]:
# Subtract the background
si_peaks["corrected_intensity"] = si_peaks["intensity"] - vals

In [25]:
fig_corrected = px.line(si_peaks, x="energy", y="corrected_intensity")    
fig_corrected.update_xaxes(autorange="reversed")

<div class="alert alert-block alert-warning">
<strong>Exercise</strong>

<p>
Read in data for the oxygen peak and perform a background correction.
</p>

<p>
The data for the oxygen peak is in the file <code>si_data/N0125704.asc</code>.
</p>

<p>
Be sure to save oxygen data in appropriate variable names. 
You will use both O and Si data later.
</p>
</div>




In [26]:
o_peaks = pd.read_table(ROOT / "si_data/N0125704.asc", header=None, names=["energy", "intensity"])
o_peaks.head()

,energy,intensity
0,544.000,8847.9422
1,543.875,8855.1659
2,543.750,8822.7821
3,543.625,8804.8229
4,543.500,8678.1587


In [27]:
o_fig = px.line(o_peaks, x="energy", y="intensity", title="Oxygen spectrum and selected background")
o_fig.update_xaxes(autorange="reversed", title="Binding energy (eV)")
o_fig.update_yaxes(title="Intensity (source units)")
o_fig.show()

In [28]:
o_background = o_peaks[(o_peaks["energy"] < 529) | (o_peaks["energy"] > 538)]
o_background.head()

,energy,intensity
0,544.000,8847.9422
1,543.875,8855.1659
2,543.750,8822.7821
3,543.625,8804.8229
4,543.500,8678.1587


**Background choice:** The intervals below 529 eV and above 538 eV exclude the main oxygen feature. This is an illustrative linear baseline; compare it visually with the measured endpoints.

In [29]:
o_background_fit = np.polyfit(o_background["energy"], o_background["intensity"], 1)
o_baseline = np.polyval(o_background_fit, o_peaks["energy"])

In [30]:
o_fig.add_scatter(x=o_background["energy"], y=o_background["intensity"], mode="markers", name="Background points")
o_fig.add_scatter(x=o_peaks["energy"], y=o_baseline, mode="lines", line_dash="dash", name="Linear background")
o_fig.show()

In [31]:
o_peaks["corrected_intensity"] = o_peaks["intensity"] - o_baseline
o_corrected_fig = px.line(o_peaks, x="energy", y="corrected_intensity", title="Oxygen spectrum after background subtraction", labels={"energy": "Binding energy (eV)", "corrected_intensity": "Corrected intensity (source units)"})
o_corrected_fig.update_xaxes(autorange="reversed")
o_corrected_fig.show()

## Numerical Integration of Peaks

Although it is possible to fit a  function to each peak and perform  integration,
we can also analyze our spectrum by doing numerical integration.

SciPy offers a number of functions for [doing numerical integration](https://docs.scipy.org/doc/scipy/tutorial/integrate.html) (see "Methods for Integrating Functions given fixed samples"). 
We will somewhat arbitrarily use `simpson`.
We imported this at the beginning of the notebook with `from scipy.integrate import simpson`.
As a reminder, our peaks look like the following:


In [32]:
fig_corrected = px.line(si_peaks, x="energy", y="corrected_intensity")    
fig_corrected.update_xaxes(autorange="reversed")

To use [`simpson` for integration](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.simpson.html#scipy.integrate.simpson), you first pass in your `y` values, then your `x` values.

In [33]:
si_peak_area = simpson(y=si_peaks["corrected_intensity"], x=si_peaks["energy"])
print(si_peak_area)

-70753.53302458854


Notice that our area is negative. 
This is because our energies are listed in descending order (highest to lowest).
This affects the sign when we are doing numerical integration.
We can fix this by sorting our dataframe based on the energy.

In [34]:
si_peaks.sort_values(by="energy", inplace=True)
si_peaks.head()

,energy,intensity,corrected_intensity
160,92.000,634.7571,392.309105
159,92.125,627.9534,368.283834
158,92.250,608.5805,331.689364
157,92.375,599.8107,305.697993
156,92.500,600.6060,289.271723


Now we can repeat the integration and get the correct sign.

In [35]:
si_peak_area = simpson(y=si_peaks["corrected_intensity"], x=si_peaks["energy"])
print(si_peak_area)

70753.53302458856


<div class="alert alert-block alert-warning">
<strong>Exercise</strong>

<p>
Repeat the integration for the oxygen peak.
</p>

<p>
After you have calculated the area of the oxygen peak, figure out the ratio of Si to O in you material by taking the ratio of the calculated areas.
</p>
</div>

In [36]:
o_peaks = o_peaks.sort_values("energy")
o_peak_area = simpson(y=o_peaks["corrected_intensity"], x=o_peaks["energy"])
print("O integrated area:", o_peak_area)

O integrated area: 135583.24309687433


In [37]:
si_to_o_area_ratio = si_peak_area / o_peak_area
print("Si/O raw area ratio:", si_to_o_area_ratio)

Si/O raw area ratio: 0.5218457045907591


In [38]:
pd.DataFrame({"Spectrum": ["Si", "O"], "Numerical area": [si_peak_area, o_peak_area]})

,Spectrum,Numerical area
0,Si,70753.533025
1,O,135583.243097


In [39]:
print("O background slope and intercept:", o_background_fit)

O background slope and intercept: [    62.47875665 -25612.17380986]


In [40]:
o_peaks[["energy", "intensity", "corrected_intensity"]]

,energy,intensity,corrected_intensity
176,522.000,7399.8304,398.093237
175,522.125,7233.5145,223.967493
174,522.250,7240.2299,222.873048
173,522.375,7296.9571,271.790404
172,522.500,7277.5399,244.563359
...,...,...,...
4,543.500,8678.1587,333.128269
3,543.625,8804.8229,451.982625
2,543.750,8822.7821,462.131980
1,543.875,8855.1659,486.705936


## Fitting Peaks with Gaussians

The following is supplemental material. We will fit a model to a background-corrected peak and integrate that model, as an alternative to integrating the measured values directly.

A Gaussian function has the form

$$
f(x) = A \exp \left( -\frac{(x - B)^2}{2C^2} \right)
$$

Here, $A$ is the peak height, $B$ is the peak center, and $C$ is the standard deviation, which controls the peak width.

We will use SciPy's [`curve_fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html), imported at the start of this notebook, to estimate these parameters from our data.
The fitting function takes the model, x values, and y values as inputs. We first need to define the model as a Python function.

The cell below defines `gaussian` and `double_gaussian`. The second function adds two Gaussians, each with its own height, center, and width, so it can describe overlapping components.
These functions use NumPy (`np`), which we also imported at the start of this notebook.


In [41]:
def gaussian(x, A, B, C):
    y = A*np.exp(-(x-B)**2/(2*C**2)) 
    return y

def double_gaussian(x, a1, b1, c1, a2, b2, c2):
    g1 = a1 * np.exp(-(x - b1)**2 / (2 * c1**2))
    g2 = a2 * np.exp(-(x - b2)**2 / (2 * c2**2))
    return g1 + g2

To peform our fit, we'll first need to pick a peak of interest.
We will do this in a way that is very similar to what we did for the background. 
This time, we can use `between`

In [42]:
# Pick a peak 
peak_1 = si_peaks[si_peaks["energy"].between(99.25, 101.5)]
peak_1.head()

,energy,intensity,corrected_intensity
102,99.250,5402.0740,4160.774913
101,99.375,9914.7366,8656.215942
100,99.500,16621.3370,15345.594772
99,99.625,24149.5770,22856.613201
98,99.750,30116.1840,28805.998631


In [43]:
# visualize the peak
fig_narrow = px.line(peak_1, x="energy", y="corrected_intensity")
fig_narrow.update_xaxes(autorange="reversed")
fig_narrow.show()

We will model this asymmetric, background-corrected peak as the sum of two Gaussians.

In `curve_fit`, the first argument is the model function, followed by the energy values and the corrected intensities.
The `p0` list contains initial guesses in the same order as the model parameters: height, center, and width for the first Gaussian, then for the second.
We start the centers at 100.0 and 100.5 eV, with heights of 30,000 and 15,000 and widths of 0.1 eV.
Using different starting centers helps the fit distinguish the two components.

The `bounds` argument contains a list of lower limits followed by a list of upper limits, in the same parameter order.
For this example, we require nonnegative heights, centers within the selected energy window, and widths between 0.01 and 1 eV.
`np.inf` leaves the peak heights without an upper limit. These limits are choices for this example, not universal settings for XPS spectra.


In [44]:
parameters_1, covariance_1 = curve_fit(
    double_gaussian,
    peak_1["energy"],
    peak_1["corrected_intensity"],
    p0=[30000, 100.0, 0.1, 15000, 100.5, 0.1],
    bounds=(
        [0, 99.25, 0.01, 0, 99.25, 0.01],
        [np.inf, 101.5, 1, np.inf, 101.5, 1],
    ),
)


In [45]:
print(parameters_1)

[1.31436485e+04 9.98052367e+01 2.60271405e-01 2.72079675e+04
 1.00308608e+02 5.32685775e-01]


We can compare our model to our actual peak by putting our x values into our gaussian function, along with our peak parameters.
In the cell below, we use `*parameters_1`. 
When we use this with a list, it fills in each element into the function arguments.

In [46]:
# Get fit values
fit_y = double_gaussian(si_peaks["energy"], *parameters_1)

Taking a wider view, we can compare the fitted first peak with the background-corrected silicon spectrum. The model was fitted only to the selected energy window.


In [47]:
fig_fit = px.line(si_peaks, x="energy", y="corrected_intensity")
fig_fit.add_scatter(x=si_peaks["energy"], y=fit_y)
fig_fit.update_xaxes(autorange="reversed")
fig_fit.show()

## Peak Integration

Now that we have our peak, we can integrate it using SciPy.
We will use [the `quad` function](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html) we imported at the top of the file.
In the quad function, you first pass the function you would like to integrate, followed by the lower and upper integration bounds.
In our case, we need to pass in our fit parameters as arguments to the `double_gaussian` function as well. That is done with the 
syntax `args=(*parameters_1,)`.

In [48]:
result, error = quad(double_gaussian, 98, 102, args=(*parameters_1,))

In [49]:
result

44876.80986335462

<div class="alert alert-block alert-warning">
<strong>Exercise</strong>

<p>
Repeat the analysis for the second Si peak and for the O peak.
</p>
</div>

In [50]:
peak_2 = si_peaks[si_peaks["energy"].between(102, 106.5)]
parameters_2, covariance_2 = curve_fit(gaussian, peak_2["energy"], peak_2["corrected_intensity"], p0=[peak_2["corrected_intensity"].max(), 104, 0.7], bounds=([0, 102, 0.01], [np.inf, 106.5, 3]))
print("Second Si peak parameters:", parameters_2)

Second Si peak parameters: [1.25415441e+04 1.04134654e+02 7.92619998e-01]


In [51]:
o_fit_region = o_peaks[o_peaks["energy"].between(529, 538)]
parameters_o, covariance_o = curve_fit(gaussian, o_fit_region["energy"], o_fit_region["corrected_intensity"], p0=[o_fit_region["corrected_intensity"].max(), 533.4, 0.7], bounds=([0, 529, 0.01], [np.inf, 538, 3]))
print("O peak parameters:", parameters_o)

O peak parameters: [7.18267350e+04 5.33456578e+02 7.32167038e-01]


In [52]:
fig_si2 = px.line(peak_2, x="energy", y="corrected_intensity", title="Second silicon peak and Gaussian fit")
fig_si2.add_scatter(x=peak_2["energy"], y=gaussian(peak_2["energy"], *parameters_2), mode="lines", line_dash="dash", name="Gaussian model")
fig_si2.update_xaxes(autorange="reversed", title="Binding energy (eV)")
fig_si2.update_yaxes(title="Corrected intensity (source units)")
fig_si2.show()

In [53]:
fig_o_fit = px.line(o_fit_region, x="energy", y="corrected_intensity", title="Oxygen peak and Gaussian fit")
fig_o_fit.add_scatter(x=o_fit_region["energy"], y=gaussian(o_fit_region["energy"], *parameters_o), mode="lines", line_dash="dash", name="Gaussian model")
fig_o_fit.update_xaxes(autorange="reversed", title="Binding energy (eV)")
fig_o_fit.update_yaxes(title="Corrected intensity (source units)")
fig_o_fit.show()

In [54]:
si_area_1, si_error_1 = quad(double_gaussian, 98, 102, args=tuple(parameters_1))
si_area_2, si_error_2 = quad(gaussian, 102, 108, args=tuple(parameters_2))
o_fit_area, o_fit_error = quad(gaussian, 529, 538, args=tuple(parameters_o))
print("Fitted areas (Si first, Si second, O):", si_area_1, si_area_2, o_fit_area)

Fitted areas (Si first, Si second, O): 44876.80986335462 24829.39183296293 131821.49477068605


In [55]:
fitted_area_ratio = (si_area_1 + si_area_2) / o_fit_area
print("Si/O fitted area ratio:", fitted_area_ratio)
print("Si/O numerical area ratio:", si_to_o_area_ratio)

Si/O fitted area ratio: 0.5287923780380205
Si/O numerical area ratio: 0.5218457045907591


In [56]:
pd.DataFrame({"Peak": ["Si first", "Si second", "O"], "Fitted area": [si_area_1, si_area_2, o_fit_area], "Quadrature error": [si_error_1, si_error_2, o_fit_error]})

,Peak,Fitted area,Quadrature error
0,Si first,44876.809863,9.711390e-05
1,Si second,24829.391833,7.188607e-08
2,O,131821.494771,1.486442e-09


<div class="alert alert-block alert-warning">
<strong>Final Exercise: Create Your Own Notebook</strong>

<p>
<strong>Create a new notebook</strong> named <code>05-xps-exercise.ipynb</code> in the same folder as the lesson notebooks. Complete this exercise there.
</p>
<p>
Start with a Markdown cell containing a title and a brief description of the problem. Add code cells for your setup and analysis. Import the pandas, NumPy, Plotly, and SciPy functions you need. Load the oxygen spectrum from <code>si_data/N0125704.asc</code> and the silicon spectrum from <code>si_data/N0125702.asc</code>. Both are tab-separated files without headers; name their columns <code>energy</code> and <code>intensity</code>. Include the Gaussian function definitions you use; you may copy those definitions from the lesson. Recreate the background correction, fits, and integrated areas for both spectra so your comparison can run entirely in this notebook.
</p>
<p>
Each notebook needs its own setup: variables and imports from the lesson notebook are not automatically available in your new notebook.
</p>


<p>
Repeat the full analysis starting from background subtraction for the oxygen peak.
</p>

<p>
Once you have fit and integrated the oxygen peak, you can compare the ratio of Si to O in the material by
taking the ratio of the area of the silicon peaks to the oxygen peak.
</p>

<p>
Finish with a Markdown cell explaining your results. Save your notebook, restart its kernel, and run all cells from top to bottom. Check that it completes without errors and reproduces your results. This checks that every step needed for your analysis is included in the notebook.
</p>
</div>

**Worked solution:** [05-xps-exercise.ipynb](05-xps-exercise.ipynb). This separate notebook includes all required setup and interpretation.

**Instructor interpretation:** Use oxygen energies below 529 eV and above 538 eV as one plausible background selection. Inspect the overlaid points and fitted line before subtraction; different endpoint choices change the integrated areas. The corrected oxygen peak is near 533.4 eV. A single Gaussian is a teaching approximation; inspect deviations between model and data. The first silicon feature uses the lesson's double Gaussian and the second uses a single Gaussian.

The printed Si/O values are **raw intensity-area ratios**. Atomic ratios require appropriate relative sensitivity factors and compatible acquisition settings; the silicon substrate and oxide also contribute differently. Do not infer SiO₂ stoichiometry directly from these ratios. Numerical integration and fitted integration use different windows and describe the tails differently. Quadrature error estimates numerical integration error only, excluding fit and background uncertainty.

References: [Thermo Fisher oxygen XPS reference](https://www.thermofisher.cn/hk/zt/home/materials-science/learning-center/periodic-table/non-metal/oxygen.html); [Avoiding common errors in XPS analysis](https://doi.org/10.1016/j.apsadv.2023.100534).

The tables below provide the measured and corrected values underlying the spectra.

In [57]:
display(si_peaks)
display(o_peaks)

,energy,intensity,corrected_intensity
160,92.000,634.7571,392.309105
159,92.125,627.9534,368.283834
158,92.250,608.5805,331.689364
157,92.375,599.8107,305.697993
156,92.500,600.6060,289.271723
...,...,...,...
4,111.500,3043.9405,114.927500
3,111.625,3037.6669,91.432329
2,111.750,3138.2208,174.764658
1,111.875,3229.2760,248.598288


,energy,intensity,corrected_intensity
176,522.000,7399.8304,398.093237
175,522.125,7233.5145,223.967493
174,522.250,7240.2299,222.873048
173,522.375,7296.9571,271.790404
172,522.500,7277.5399,244.563359
...,...,...,...
4,543.500,8678.1587,333.128269
3,543.625,8804.8229,451.982625
2,543.750,8822.7821,462.131980
1,543.875,8855.1659,486.705936
